# Complete ML + Student Performance Model

## Overview
This notebook walks through the complete machine learning pipeline for predicting student performance using AWS SageMaker:
- Data loading and preprocessing
- Model training using Random Forest Regression
- Model evaluation and validation
- Uploading model artifacts to S3
- Deploying a SageMaker endpoint for inference
- Testing the endpoint
- Cleaning up resources

## Prerequisites
- Python 3.7+
- AWS account with SageMaker access
- AWS credentials configured locally
- `Student_Performance.csv` file in the same directory as this notebook
- Budget alerts set up on your AWS account (endpoint deployment incurs hourly costs)

## ⚠️ IMPORTANT REMINDERS
1. **Endpoint Deployment (Cell 13):** Creates a live AWS endpoint that incurs costs (~$0.115/hour for ml.m5.large)
2. **Endpoint Cleanup (Cell 15):** MUST be run to delete the endpoint and stop charges
3. **CSV Requirement:** Ensure `Student_Performance.csv` exists and contains a column named "Performance Index"
4. **Execution Order:** Run cells sequentially from top to bottom

## 📦 1. Install & Import

In [3]:
import pandas as pd
import numpy as np
import sagemaker
import boto3
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
print(sagemaker)

## 📊 2. Load Dataset

In [ ]:
df = pd.read_csv("Student_Performance (2).csv")
df.head()

In [ ]:
df.shape

## 🔍 3. Data Preprocessing

In [ ]:
# Check missing values
print(df.isnull().sum())

# Encode categorical columns
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

df.head()

## 🎯 4. Define Features & Target

In [8]:
X = df.drop("overall_score", axis=1)
y = df["overall_score"]

## ✂️ 5. Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 🧠 6. Train Model (Regression)

In [10]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


## 📈 7. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

## 💾 8. Save Model

In [ ]:
joblib.dump(model, "model.joblib")
print("Model saved!")

## ☁️ 9. Upload to S3

In [ ]:
import boto3

bucket = "Your Bucket Name"

s3 = boto3.client("s3")

with open("model.joblib", "rb") as f:
    s3.put_object(
        Bucket=bucket,
        Key="ml-project/model.joblib",
        Body=f
    )

print("Model Uploaded Successfully!")

## ⚙️ 10. Inference Script

This cell creates an `inference.py` file using the `%%writefile` IPython magic command.

In [ ]:
%%writefile inference.py

import joblib
import numpy as np

def model_fn(model_dir):
    model = joblib.load(f"{model_dir}/model.joblib")
    return model

def predict_fn(input_data, model):
    input_data = np.array(input_data).reshape(1, -1)
    prediction = model.predict(input_data)
    return prediction.tolist()

## 📦 11. Package Model

In [ ]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib")
    tar.add("inference.py")

print("Model packaged!")

## ☁️ 12. Upload Model Artifact

In [ ]:
import boto3

s3 = boto3.client("s3")

bucket = "Your Bucket Name"

with open("model.tar.gz", "rb") as f:
    s3.put_object(
        Bucket=bucket,
        Key="ml-project/model.tar.gz",
        Body=f
    )

model_artifact = f"s3://{bucket}/ml-project/model.tar.gz"

print(model_artifact)

## 🚀 13. Deploy Endpoint

⚠️ **WARNING: COSTS MONEY** ⚠️

This cell will create a live SageMaker endpoint that incurs AWS charges:
- **Instance type:** `ml.m5.large` (~$0.115/hour)
- **Cost calculation:** Instance runs 24/7 until you delete it (see Cell 15)
- **Example:** Running for 24 hours = ~$2.76

**IMPORTANT:** You MUST run Cell 15 to delete the endpoint and stop charges.

In [ ]:
!pip install sagemaker scikit-learn --upgrade

In [ ]:
!pip uninstall -y sagemaker
!pip install sagemaker==2.245.0

In [ ]:
from sagemaker.sklearn.model import SKLearnModel

print("Import Successful!")

In [ ]:
from sagemaker.sklearn.estimator import SKLearn
import sagemaker

role = sagemaker.get_execution_role()

sk_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3"
)

predictor = sk_model.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1
)

print("Endpoint deployed!")

## 🧪 14. Test Endpoint

In [ ]:
sample = X_test.iloc[0:1]

predicted_score = model.predict(sample)[0]

actual_score = y_test.iloc[0]

print("Actual Overall Score:", actual_score)
print("Predicted Overall Score:", predicted_score)

## 🛑 15. Delete Endpoint

⚠️ **CRITICAL: RUN THIS TO STOP CHARGES** ⚠️

If you skip this step, the endpoint will continue to run and incur charges until you manually delete it in the AWS console.

**Always run this cell** when you're done testing to avoid unexpected AWS bills.

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted!")